### Convert html file to markdown with @postlight/parser

Snce NotebookLM accepts markdown, and since no matter what you give it, the first thing it does is to strip out anything but text, this seems to be a good solution.

[Perplexity recommended]{https://www.perplexity.ai/search/the-obsidian-web-clipper-exten-J473Sf_OTAaeDiXrItIgGg#3} as an alternative to trying to get the obsidian web clipper extension to work

#### Installation

`npm install -g @postlight/parser`

#### Results
- does work when downloading html from a URL and then writing markdown
- but I can't get it to work when it reads a local html file
- a real PTA to install, as it runs by calling a Node.js script

In [15]:
import pathlib as pl
import sys
from icecream import ic

refwrangle_dir = pl.Path('~/ref/refwrangle').expanduser()
sys.path.append(str(refwrangle_dir))
import refwrangle as rfw # assumed to be in same dir as this

html_input_dir = pl.Path(r'C:/Users/scott/OneDrive/share/ref/obsidian/Obsidian Share Vault/lit/lit_sources/')

html_file_path = html_input_dir / 'Tumulty24FrischLearnedDemsShould.html'  # 
#html_file_path = html_input_dir / 'Walther24barstoolConservatism.html'     # 
#html_file_path = html_input_dir / 'Yan24berkeleyFuncCallLeaderBrd.html'     # 

output_md_path = refwrangle_dir / "test/tmp_obswebclip.md"

### Calling via running a node script that downloaded a URL via subprocess worked

But 
- I was almost able to use @postlight/parser's commandline postlight-parser by calling it with a subprocess.  However, the output file had some encoding problem that I could not figure out, nor could complexity
- So I tried subprocessing a node script that called @postlight/parser directly
- To get this to work, I had to install @postlight/parser in the same directory as this .ipynb
- I could find no way to make this POS handle windows paths.
  - it couldn't find an html input file because it couldn't do anyting with a Windows path
- I *could* write to an output path using a pretend unix path relevant to the cwd() e.g. './tmp/output.md'

In [ ]:
import subprocess
import sys
from pathlib import Path

def parse_url_to_markdown(url, output_file):
    # Use proper Node.js require syntax
    node_script = (
        'const Parser = require("@postlight/parser");'
        f'Parser.parse("{url}", {{contentType: "markdown"}})'
        f'.then(result => {{'
        f'  require("fs").writeFileSync("{output_file}", result.content, "utf8");'
        f'}})'
        f'.catch(error => {{'
        f'  console.error(error);'
        f'  process.exit(1);'
        f'}});'
    )
    
    try:
        # Run Node.js in the project directory where node_modules exists
        project_dir = Path.cwd()
        subprocess.run(
            ["node", "-e", node_script],
            check=True,
            text=True,
            encoding='utf-8',
            stderr=subprocess.PIPE,
            cwd=str(project_dir),
            shell=True
        )
        print(f"Successfully saved markdown to {output_file}")
    except subprocess.CalledProcessError as e:
        print(f"Error occurred: {e.stderr}", file=sys.stderr)
        raise

# WORKS
#url = "https://example.com"
# WORKS
url = 'https://www.nytimes.com/interactive/2025/01/14/opinion/fix-congress-proportional-representation.html'  # tons of animation
# FAILS b/c of WApost paywall?
#url = 'https://www.washingtonpost.com/opinions/2024/12/01/adam-frisch-democrats-lauren-boebert/' # weird WAPost stuff

# WORKS:
output_file = "output.md"  # WORKS
# output_file = "./tmp/output.md" # WORKS: fake relative unix path works
# output_file = "C:/Users/scott/ref/refwrangle/test/tmp/output.md" # WORKS: fake unix path works w/ drive letter works
#
# BREAKS
# import urllib.parse 
# output_file = urllib.parse.urljoin('file:', urllib.parse.quote(str(output_md_path.absolute())))

ic(output_file)

parse_url_to_markdown(url, str(output_file))

## Another try at files

In [ ]:
# making node.js compatible paths

from pathlib import Path

def convert_to_nodejs_path(path_input):
    # Convert input to Path object if it's a string
    path = Path(path_input) if isinstance(path_input, str) else path_input
    # Convert to string and replace backslashes with forward slashes
    return str(path).replace('\\', '/')

# Test cases
test_paths = [
    "C:\\Users\\Example\\file.txt",
    "\\\\network\\share\\file.txt",
    ".\\relative\\path.txt",
    Path("D:\\Projects\\test.js"),
    "C:\\Program Files\\App\\config.json"
]

for path in test_paths:
    print(f"Original: {path}")
    print(f"Converted: {convert_to_nodejs_path(path)}\n")


In [ ]:
import subprocess
import sys
from pathlib import Path
import json

def parse_html_to_markdown(input_file, output_file):
    input_path = str(Path(input_file).resolve())
    output_path = str(Path(output_file).resolve())
    
    node_script = (
        'const Parser = require("@postlight/parser");'
        'const fs = require("fs");'
        f'const htmlContent = fs.readFileSync({json.dumps(input_path)}, "utf8");'
        'Parser.parse(htmlContent, {contentType: "markdown"})'
        '.then(result => {'
        '    if (!result || !result.content) {'
        '        console.error("Parser returned no content");'
        '        process.exit(1);'
        '    }'
        f'    fs.writeFileSync({json.dumps(output_path)}, result.content, "utf8");'
        '    console.log("Parsing completed successfully");'
        '})'
        '.catch(error => {'
        '    console.error("Parsing failed:", error);'
        '    process.exit(1);'
        '});'
    )
    
    try:
        project_dir = Path.cwd()
        result = subprocess.run(
            ["node", "-e", node_script],
            check=True,
            text=True,
            encoding='utf-8',
            stderr=subprocess.PIPE,
            stdout=subprocess.PIPE,
            cwd=str(project_dir),
        )
        print(f"Successfully saved markdown to {output_path}")
        if result.stdout:
            print("Output:", result.stdout)
    except subprocess.CalledProcessError as e:
        print(f"Error occurred: {e.stderr}", file=sys.stderr)
        raise



#output_file = "C:/Users/scott/ref/refwrangle/test/tmp/output.md" # WORKS: fake unix path works w/ drive letter works
output_file = "output.md" # WORKS: fake unix path works w/ drive letter works
#input_file = "C:\Users\scott\OneDrive\share\ref\obsidian\Obsidian Share Vault\lit\lit_sources\Blake25infightMAGA.html"
#input_file = "C:/Users/scott/OneDrive/share/ref/obsidian/Obsidian Share Vault/lit/lit_sources/Blake25infightMAGA.html"
input_file = "./Blake25infightMAGA.html"
parse_html_to_markdown(input_file, output_file)


### hacked original broken file to file

In [ ]:

import subprocess
import sys
from pathlib import Path

def parse_html_to_markdown(input_file, output_file):
    # Convert paths to absolute paths to avoid any path issues
    input_path = Path(input_file).resolve()
    output_path = Path(output_file).resolve()

    ic(json.dumps(input_path))
    
    node_script = (
        'const Parser = require("@postlight/parser");'
        f'const fs = require("fs");'
        f'const htmlContent = fs.readFileSync("{json.dumps(input_path)}", "utf8");'
        f'Parser.parse(htmlContent, {{contentType: "markdown"}})'
        f'.then(result => {{'
        f'  fs.writeFileSync("{json.dumps(output_path)}", result.content, "utf8");'
        f'}})'
        f'.catch(error => {{'
        f'  console.error(error);'
        f'  process.exit(1);'
        f'}});'
    )
    
    try:
        project_dir = Path.cwd()
        subprocess.run(
            ["node", "-e", node_script],
            check=True,
            text=True,
            encoding='utf-8',
            stderr=subprocess.PIPE,
            cwd=str(project_dir),
            shell=True
        )
        print(f"Successfully saved markdown to {output_path}")
    except subprocess.CalledProcessError as e:
        print(f"Error occurred: {e.stderr}", file=sys.stderr)
        raise


#output_file = "C:/Users/scott/ref/refwrangle/test/tmp/output.md" # WORKS: fake unix path works w/ drive letter works
output_file = "output.md" # WORKS: fake unix path works w/ drive letter works
#input_file = "C:\Users\scott\OneDrive\share\ref\obsidian\Obsidian Share Vault\lit\lit_sources\Blake25infightMAGA.html"
#input_file = "C:/Users/scott/OneDrive/share/ref/obsidian/Obsidian Share Vault/lit/lit_sources/Blake25infightMAGA.html"
input_file = "./Blake25infightMAGA.html"
parse_html_to_markdown(input_file, output_file)    

In [ ]:
# # a fix to perplexity's json.dumps() bug

# force nodejs.paths
import subprocess
import sys
from pathlib import Path
import json
import os

def parse_html_to_markdown(input_file, output_file):
    # Get the directory containing the notebook
    notebook_dir = Path().absolute()
    
    # Convert paths to absolute paths relative to notebook directory
    input_path = str(Path(notebook_dir / input_file).resolve())
    output_path = str(Path(notebook_dir / output_file).resolve())
    
    # Ensure node_modules is in the notebook directory
    if not (notebook_dir / 'node_modules' / '@postlight').exists():
        print(f"Cannot find @postlight/parser in {notebook_dir}")
        return
        
    node_script = (
        'const Parser = require("@postlight/parser");'
        'const fs = require("fs");'
        f'const htmlContent = fs.readFileSync({json.dumps(input_path)}, "utf8");'
        'Parser.parse(htmlContent, {contentType: "markdown"})'
        '.then(result => {'
        '    if (!result || !result.content) {'
        '        console.error("Parser returned no content");'
        '        process.exit(1);'
        '    }'
        f'    fs.writeFileSync({json.dumps(output_path)}, result.content, "utf8");'
        '    console.log("Parsing completed successfully");'
        '})'
        '.catch(error => {'
        '    console.error("Parsing failed:", error);'
        '    process.exit(1);'
        '});'
    )

    ic(node_script)
    
    try:
        # Use notebook directory as working directory
        result = subprocess.run(
            ["node", "-e", node_script],
            check=True,
            text=True,
            encoding='utf-8',
            stderr=subprocess.PIPE,
            stdout=subprocess.PIPE,
            cwd=str(notebook_dir)
        )
        print(f"Successfully saved markdown to {output_path}")
        if result.stdout:
            print("Output:", result.stdout)
    except subprocess.CalledProcessError as e:
        print(f"Error occurred: {e.stderr}", file=sys.stderr)
        raise


#output_file = "C:/Users/scott/ref/refwrangle/test/tmp/output.md" # WORKS: fake unix path works w/ drive letter works
output_file = "output.md" # WORKS: fake unix path works w/ drive letter works
#input_file = "C:\Users\scott\OneDrive\share\ref\obsidian\Obsidian Share Vault\lit\lit_sources\Blake25infightMAGA.html"
#input_file = "C:/Users/scott/OneDrive/share/ref/obsidian/Obsidian Share Vault/lit/lit_sources/Blake25infightMAGA.html"
input_file = "./Blake25infightMAGA.html"
parse_html_to_markdown(input_file, output_file)

ic| node_script: ('const Parser = require("@postlight/parser");const fs = require("fs");const '
                  'htmlContent = '
                  'fs.readFileSync("C:\\\\Users\\\\scott\\\\OneDrive\\\\share\\\\ref\\\\refwrangle\\\\test\\\\Blake25infightMAGA.html", '
                  '"utf8");Parser.parse(htmlContent, {contentType: "markdown"}).then(result => '
                  '{    if (!result || !result.content) {        console.error("Parser returned '
                  'no content");        process.exit(1);    }    '
                  'fs.writeFileSync("C:\\\\Users\\\\scott\\\\OneDrive\\\\share\\\\ref\\\\refwrangle\\\\test\\\\output.md", '
                  'result.content, "utf8");    console.log("Parsing completed '
                  'successfully");}).catch(error => {    console.error("Parsing failed:", '
                  'error);    process.exit(1);});')
Error occurred: Parser returned no content



CalledProcessError: Command '['node', '-e', 'const Parser = require("@postlight/parser");const fs = require("fs");const htmlContent = fs.readFileSync("C:\\\\Users\\\\scott\\\\OneDrive\\\\share\\\\ref\\\\refwrangle\\\\test\\\\Blake25infightMAGA.html", "utf8");Parser.parse(htmlContent, {contentType: "markdown"}).then(result => {    if (!result || !result.content) {        console.error("Parser returned no content");        process.exit(1);    }    fs.writeFileSync("C:\\\\Users\\\\scott\\\\OneDrive\\\\share\\\\ref\\\\refwrangle\\\\test\\\\output.md", result.content, "utf8");    console.log("Parsing completed successfully");}).catch(error => {    console.error("Parsing failed:", error);    process.exit(1);});']' returned non-zero exit status 1.

In [19]:
print(f"Notebook directory: {Path().absolute()}")
print(f"Node modules path: {Path().absolute() / 'node_modules' / '@postlight'}")
print(f"Input file path: {Path().absolute() / input_file}")


Notebook directory: c:\Users\scott\ref\refwrangle\test
Node modules path: c:\Users\scott\ref\refwrangle\test\node_modules\@postlight
Input file path: c:\Users\scott\ref\refwrangle\test\Blake25infightMAGA.html


In [12]:
# ## original file to file w/ spaces-in-paths problem


# import subprocess
# import sys
# from pathlib import Path

# def parse_html_to_markdown(input_file, output_file):
#     # Convert paths to absolute paths to avoid any path issues
#     input_path = Path(input_file).resolve()
#     output_path = Path(output_file).resolve()
    
#     node_script = (
#         'const Parser = require("@postlight/parser");'
#         f'const fs = require("fs");'
#         f'const htmlContent = fs.readFileSync("{input_path}", "utf8");'
#         f'Parser.parse(htmlContent, {{contentType: "markdown"}})'
#         f'.then(result => {{'
#         f'  fs.writeFileSync("{output_path}", result.content, "utf8");'
#         f'}})'
#         f'.catch(error => {{'
#         f'  console.error(error);'
#         f'  process.exit(1);'
#         f'}});'
#     )
    
#     try:
#         project_dir = Path.cwd()
#         subprocess.run(
#             ["node", "-e", node_script],
#             check=True,
#             text=True,
#             encoding='utf-8',
#             stderr=subprocess.PIPE,
#             cwd=str(project_dir),
#             shell=True
#         )
#         print(f"Successfully saved markdown to {output_path}")
#     except subprocess.CalledProcessError as e:
#         print(f"Error occurred: {e.stderr}", file=sys.stderr)
#         raise


# #output_file = "C:/Users/scott/ref/refwrangle/test/tmp/output.md" # WORKS: fake unix path works w/ drive letter works
# output_file = "output.md" # WORKS: fake unix path works w/ drive letter works
# #input_file = "C:\Users\scott\OneDrive\share\ref\obsidian\Obsidian Share Vault\lit\lit_sources\Blake25infightMAGA.html"
# #input_file = "C:/Users/scott/OneDrive/share/ref/obsidian/Obsidian Share Vault/lit/lit_sources/Blake25infightMAGA.html"
# input_file = "./Blake25infightMAGA.html"
# parse_html_to_markdown(input_file, output_file)    

### Input html file doesn't work

A Windows path disaster

In [ ]:
import subprocess
import sys
from pathlib import Path
from urllib.parse import urljoin
from urllib.request import pathname2url

def parse_html_to_markdown(input_file, output_file):
    # Convert local path to file URI
    input_path = Path(input_file).resolve()
    if not input_path.exists():
        raise FileNotFoundError(f"Input file not found: {input_path}")
    
    # Create file URI (e.g., file:///C:/Users/...)
    file_uri = 'file:///' + pathname2url(str(input_path))
    output_path = Path(output_file).resolve()
    
    node_script = (
        'const Parser = require("@postlight/parser");'
        f'Parser.parse("{file_uri}", {{contentType: "markdown"}})'
        f'.then(result => {{'
        f'  require("fs").writeFileSync("{str(output_path).replace("\\", "/")}", result.content, "utf8");'
        f'}})'
        f'.catch(error => {{'
        f'  console.error(error);'
        f'  process.exit(1);'
        f'}});'
    )
    
    try:
        project_dir = Path.cwd()
        subprocess.run(
            ["node", "-e", node_script],
            check=True,
            text=True,
            encoding='utf-8',
            stderr=subprocess.PIPE,
            cwd=str(project_dir),
            shell=False
        )
        print(f"Successfully saved markdown to {output_path}")
    except subprocess.CalledProcessError as e:
        print(f"Error occurred: {e.stderr}", file=sys.stderr)
        raise

parse_html_to_markdown(html_file_path, output_md_path)